# Session 2 · Cleansing

**Goal:** load the file the right way, then fix whitespace, codes that mean "unknown," damaged text, names, keys, coded values (race, religion, STG affiliation, offense and violence codes), and multi-value fields. Save the result in a format that keeps what we fixed.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 2

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import check, cleaning, domains
from lnl.check import ___

## 1. Load it the right way
Three settings make the difference:
- `dtype=str` reads **every** column as text. Nothing gets "helpfully" converted, so TDCJ and SID numbers arrive exactly as the source wrote them, leading zeros included, ready to be checked.
- `keep_default_na=False` and `na_filter=False` stop pandas from deciding what counts as missing. By default it treats `NA`, `N/A`, `null`, `None` and more as missing, which is a decision that belongs to us and the data owner.

In [ ]:
df = pd.read_csv(PATHS["data"], dtype=str, keep_default_na=False, na_filter=False)

print("TDCJ #02381457 exists:", (df["tdcj_number"] == "02381457").any())    # text search, text column: found
df[["tdcj_number", "sid_number"]].head()

## 2. Strip and null
Last session's fix, now saved as a reusable function (`cleaning.strip_and_null`) so it's applied exactly the same way every time.

In [ ]:
df = cleaning.strip_and_null(df)
df.isna().sum().to_frame("missing values")

## 3. Make `tdcj_number` the record ID
Every record has exactly one TDCJ number, never repeated, and the source writes it as 8-digit text with leading zeros: `02381457`. In our tables, TDCJ and SID numbers are stored as whole numbers (**BIGINT**): compact, fast to join, and the same type in every table. The rule is **check first, then convert on purpose**:

- `cleaning.set_record_id` first **proves** the TDCJ number is a real ID: none missing, all exactly 8 digits, none repeated. If any record fails, it stops with an error instead of guessing.
- It then converts `tdcj_number` and `sid_number` to whole numbers (`Int64`, pandas' equivalent of BIGINT that allows blanks). `02381457` is now stored as `2381457`.
- Finally it makes `tdcj_number` the DataFrame's **index**, so every table from now on is labeled by TDCJ number and you can look a record up directly.

A number has no leading zeros, so two habits come with it. Look a record up with `cleaning.id_value("02381457")`, which applies the same check and conversion to the ID you typed. Show IDs to people with `cleaning.format_id()`, which pads them back to 8 digits.

We stripped whitespace first on purpose: an ID with a stray space in front would fail the check, not slip through.

In [ ]:
df = cleaning.set_record_id(df)

print("Index:", df.index.name, "| type:", df.index.dtype, "| unique:", df.index.is_unique)
print('Typed as text:  "02381457" in df.index ->', "02381457" in df.index)                        # text vs number: no match
print('Converted:      id_value("02381457") in df.index ->', cleaning.id_value("02381457") in df.index)   # Q5 is fixed
superman = cleaning.id_value("02381457")
print("Stored as", superman, "| shown to people as", cleaning.format_id(superman))
df.loc[superman, ["last_name", "first_name", "alias", "sid_number"]]

## 4. Codes that mean "unknown"
Source systems often put a placeholder in a field instead of leaving it blank. Let's look for the usual suspects:

In [ ]:
suspects = ["N/A", "U", "UNK", "99:99", "NONE"]
for col in ["disciplinary_points", "escape_risk", "protective_custody", "intake_time", "housing_restriction"]:
    counts = df[col].value_counts()
    print(f"{col:22} {counts[counts.index.isin(suspects)].to_dict()}")

Each one needs a decision, and **the data owner makes it, not the analyst**:

| Column | Code | What it means | Decision |
|---|---|---|---|
| `disciplinary_points` | `N/A` | not recorded | make it null |
| `escape_risk` | `U`, `UNK` | unknown | make it null |
| `protective_custody` | `U` | unknown | make it null |
| `intake_time` | `99:99` | the old system's "unknown time" | make it null |
| `housing_restriction` | `NONE` | **no restriction is needed** | **keep it.** It's a real answer |

The last row is the trap. `NONE` *looks* like a missing value, but it's an answer: this person needs no special housing. A blank means "we don't know." Treat them the same and every housing report is wrong.

Coded columns such as STG affiliation and religion have their own version of this decision, in section 7.

Single-character flags raise one more question for the data owner: **does a blank flag mean No** (an unchecked box) **or unknown?** In this data it means unknown. Ask; don't assume.

The decisions live in a data dictionary, so they're applied identically every time:

In [ ]:
cleaning.NULL_TOKENS

In [ ]:
before = df[list(cleaning.NULL_TOKENS)].isna().sum()
df = cleaning.apply_null_tokens(df)
after = df[list(cleaning.NULL_TOKENS)].isna().sum()
pd.DataFrame({"missing before": before, "missing after": after})

🧪 **Your turn #1.** How many `housing_restriction` values are the text `NONE`?

In [ ]:
answer_1 = ___      # compare df["housing_restriction"] to "NONE", then add up the matches with .sum()
check.s2_ex1(answer_1)

## 5. Names and free text
Before text can be matched or searched, it has to say what was actually typed. Two kinds of damage turn up in almost every source system:
- **Encoding damage** (often called *mojibake*): text saved in one character encoding and read back in another. `MUÑOZ` becomes `MUÃ‘OZ`, and an apostrophe becomes `â€™`. It usually happens when data passes through an old export or a spreadsheet.
- **Web-form leftovers:** HTML that came along with text pasted into a form. `<br>` for a line break, `&nbsp;` for a space, curly quotes from a word processor, doubled spaces.

`cleaning.repair_text_columns` fixes a fixed list of known damage (`domains.TEXT_REPAIRS`) in the name columns and the officer notes. It doesn't guess: accents that were typed on purpose, like *García*, are kept.

In [ ]:
before = df[domains.TEXT_COLUMNS].copy()
df = cleaning.repair_text_columns(df)

print("values repaired:", {col: int((before[col].notna() & before[col].ne(df[col])).sum()) for col in domains.TEXT_COLUMNS})
names = before["last_name"].str.contains("Ã", na=False)
notes = before["conduct_notes"].str.contains("<br>|&nbsp;|â€", na=False)
display(pd.DataFrame({"as written": before.loc[names, "last_name"], "repaired": df.loc[names, "last_name"]}).head(4))
pd.DataFrame({"as written": before.loc[notes, "conduct_notes"], "repaired": df.loc[notes, "conduct_notes"]}).head(4)

Now casing. Here's everyone named Kent:

In [ ]:
df.loc[df["last_name"].str.upper() == "KENT", ["last_name", "first_name", "alias", "sid_number"]]    # labeled by TDCJ number

Three records, each with its own TDCJ number: the same person, three spellings, and one record missing its SID. For **matching**, we add standardized columns: uppercase, single spaces, and accents folded, so *García*, *GARCIA* and *garcia* all match as `GARCIA`. The originals, accents included, stay for **display**.

In [ ]:
df = cleaning.standardize_names(df)
print("Records with last name KENT:", (df["last_name_std"] == "KENT").sum())    # Q3 is fixed
df.loc[df["last_name_std"] == "GARCIA", "last_name"].str.upper().value_counts().to_frame("GARCIA, as written")   # with and without the accent

Proper case for reports is harder than it looks. `str.title()`, the obvious tool, gets a lot of names wrong:

In [ ]:
examples = pd.Series(["MCDONALD", "O'CONNOR", "DE LA CRUZ", "WORTHINGTON III", "T'CHALLA", "SMITH JR", "VAN DYNE"])
pd.DataFrame({"raw": examples,
              "str.title()": examples.str.title(),
              "cleaning.display_name()": examples.map(cleaning.display_name)})

`display_name()` uses rules for Mc, apostrophes, particles like *de la*, and suffixes. No rule set is perfect (think *MacDonald* vs *Macy*), which is why **matching always uses the `_std` columns**, never the display version.

🧪 **Your turn #2.** How many different spellings of MCDONALD are in the original `last_name` column?

In [ ]:
answer_2 = ___      # filter rows where last_name_std == "MCDONALD", then .nunique() on last_name
check.s2_ex2(answer_2)

## 6. Check the keys
Before trusting any join or count, check the identifiers. In this system a person keeps the same **SID** for life but gets a **new TDCJ number each time they're incarcerated**. So a TDCJ number identifies a *record*, and a SID identifies a *person*.

In [ ]:
cleaning.key_checks(df)

What that tells us:
- TDCJ numbers are clean: none missing, none duplicated, all valid 8-digit IDs (`set_record_id` would have stopped otherwise).
- About 3,000 records have no SID, so they can't be linked to a person.
- No SID has conflicting last names (good; that would signal a bad merge).
- Thousands of people have more than one record, so **counting rows is not counting people**.

🧪 **Your turn #3.** How many people have more than one record?

In [ ]:
records_per_person = df.groupby("sid_number").size()
answer_3 = ___      # how many values in records_per_person are greater than 1?
check.s2_ex3(answer_3)

## 7. Coded values: reference data, not guesswork
Five columns hold **codes**: race, religion, security threat group (STG) affiliation, offense code, and violence code. Each has a short list of real answers and many ways to write them:

In [ ]:
pd.Series({col: df[col].nunique() for col in domains.DOMAIN_COLUMNS}, name="distinct values as written").to_frame()

The fix is a **crosswalk**: a reference table, agreed with the data owner, listing every known spelling and the standard value it stands for. Ours are CSV files in `lnl/reference/`, kept in Git and reviewed like code. The code only looks values up. It never decides what a value means.

Values are matched on a spelling-insensitive **key** (upper case, periods removed, single spaces), so `H.Y.D.R.A.`, `Hydra` and `HYDRA` are one entry, not three. Here's part of the race crosswalk:

In [ ]:
race_xw = domains.reference("race_crosswalk")
race_xw[race_xw["race_code"].isin(["W", "H"])]

Race gets **two** standardized columns:
- `race_code`: the single-letter code the agency's systems use (W, B, H, A, I, O, U, D).
- `race_omb`: the 2024 federal (OMB) categories used for federal reporting. They're more detailed: *Middle Eastern or North African* is its own category (it counted as White before 2024), *Native Hawaiian or Pacific Islander* is separate from Asian, and Hispanic or Latino can be recorded together with a race.

Some old values can't be placed in the 2024 categories without more information. `Asian/Pacific Islander` was one box on an old form, and the new standard needs to know which. The crosswalk says **Needs review** instead of guessing.

> 🔒 **Race and religion are sensitive.** They're collected for required reporting and for religious accommodations, and access to them should be restricted. Standardize what was recorded; **never infer** race or religion from a name, a photo, or anything else. In this course they're assigned at random, independently of every other column, so no pattern you find in them means anything.

### `N/A` doesn't mean the same thing everywhere
| Column | Written as | Means | Decision |
|---|---|---|---|
| `stg_affiliation` | `NONE`, `N/A`, `NO KNOWN AFFIL` | no known affiliation | a real answer: `None` |
| `religion` | `NONE`, `NO PREF` | no religious preference | a real answer |
| `religion` | `N/A`, `UNK` | not recorded | Unknown |
| `race`, `religion` | `DECLINED`, `REFUSED` | the person chose not to say | a real answer, and not the same as unknown |

Same text, different meaning per column. That's why these decisions live in each column's crosswalk, not in one global list of "null words," and why we load with `keep_default_na=False`.

`domains.standardize_domains` adds the standardized columns for all five domains. The raw columns stay exactly as written, so every mapping can be checked.

In [ ]:
df = domains.standardize_domains(df)
display(df.dropna(subset=["race"]).drop_duplicates("race")[["race", "race_code", "race_omb"]].head(12))
print('race == "W" as written:', (df["race"] == "W").sum(), '| race_code == "W":', (df["race_code"] == "W").sum())

Counting the letter `W` as written finds about half of the people recorded as White. Here's the whole population in the 2024 federal categories, including the records that need review:

In [ ]:
df["race_omb"].value_counts(dropna=False).to_frame("records")

Religion works the same way: `religion_group` is the specific faith group, and `religion_family` is a broader family for summary reports.

In [ ]:
df.dropna(subset=["religion"]).drop_duplicates("religion")[["religion", "religion_group", "religion_family"]].head(10)

### STG affiliation: two facts in one field
`stg_affiliation` holds a group **and** a status in one cell: `SUSP: H.Y.D.R.A.`, `INTERGANG (C)`, `FORMER Hellfire Club`, `Legion of Doom-RENOUNCED`. The crosswalk-driven split puts them in `stg_group` and `stg_status` (Confirmed, Suspected, Former, Unverified, or None). A bare group name with no status was never confirmed by anyone, so the data owner's rule makes it **Unverified**, not Confirmed. This matters: STG status feeds housing and classification decisions. (The groups here are fictional villain organizations.)

In [ ]:
hydra = df.loc[df["stg_group"] == "HYDRA", ["stg_affiliation", "stg_group", "stg_status"]]
display(hydra.drop_duplicates("stg_affiliation").head(10))
print('stg_affiliation == "HYDRA" as written:', (df["stg_affiliation"] == "HYDRA").sum())
print("any HYDRA affiliation:                ", len(hydra))
df["stg_status"].value_counts(dropna=False).to_frame("records, all groups")

### Offense and violence codes
Offense codes follow the FBI's NIBRS code list (`13A` is aggravated assault, `290` is destruction of property). They break in predictable ways: lower case (`13a`), extra punctuation (`13-A`, `13 A`), a spreadsheet that turned `290` into `290.0`, and a lost leading zero (`9B` for `09B`, the same problem as the IDs). `offense_code_std` holds the repaired code only if it's on the NIBRS list, and `offense_against` says whether the crime was against a person, property, or society. Some cells hold a description (`AGG ASSAULT`) instead of a code; nothing turns a description into a code automatically.

Violence codes (NV, V1, V2, V3) get the same treatment: `V-2`, `v2`, `V 2` and a bare `2` all become `V2`.

In [ ]:
pd.DataFrame({
    "as written":   [(df["offense_code"] == "13A").sum(), (df["offense_code"] == "09B").sum(), (df["violence_code"] == "V3").sum()],
    "standardized": [(df["offense_code_std"] == "13A").sum(), (df["offense_code_std"] == "09B").sum(), (df["violence_code_std"] == "V3").sum()],
}, index=["offense code 13A (aggravated assault)", "offense code 09B (negligent manslaughter)", "violence code V3"])

### What the crosswalks don't know
Anything no crosswalk recognizes goes on an **exceptions list**, keyed by TDCJ number (padded to 8 digits, because people read it). Nothing on it is guessed or dropped. `JEDI` isn't on our religion list, `V4` isn't a violence code, `ASSAULT` could be 13A or 13B, and `HYDRA/SIN6` names two groups in one cell. The data owner decides each one, and the answer goes into the crosswalk, so it's handled on every future load.

In [ ]:
unknown = domains.domain_exceptions(df)
print(f"{len(unknown):,} coded values to send back to the data owner")
display(unknown.groupby(["column", "reason"]).size().to_frame("values"))
unknown.head(8)

### Checking across records
A person (SID) can have several records, and some facts shouldn't change between them. Race is recorded once per person, so two race codes for the same SID point to a data-entry error or a bad merge: flag it, don't pick one. Religion is different. People can change faith between incarcerations, so different religions across records are **not** an error. Which facts may change is a business rule, and it comes from the data owner.

In [ ]:
conflicts = domains.race_conflicts(df)
print(f"{len(conflicts):,} people have more than one race code on file")
conflicts.head()

🧪 **Your turn #4.** How many records show a **Confirmed** HYDRA affiliation?

In [ ]:
answer_4 = ___      # two conditions: stg_group is "HYDRA" and stg_status is "Confirmed". Join them with &, then .sum()
check.s2_ex4(answer_4)

## 8. Several values in one cell
`superpower` packs several values into one cell (`FLIGHT;SUPER STRENGTH`). You can't count or filter by power properly until each one gets its own row. That's called *exploding* the column.

In [ ]:
powers = cleaning.explode_powers(df)
print(f"{len(df):,} records became {len(powers):,} (record, power) rows")
powers["power"].value_counts().head(10)

In [ ]:
print("contains 'FLIGHT':", df["superpower"].str.contains("FLIGHT", na=False).sum())
print("exactly 'FLIGHT': ", (powers["power"] == "FLIGHT").sum())

`contains` also matches `FLIGHT (WINGS)` and `FLIGHT (JETPACK)`. That may or may not be what the question means; exploded values let you choose deliberately.

## 9. Save it: Parquet, not CSV

In [ ]:
out = PATHS["dir"]
df.to_csv(out / "session2_clean.csv")          # the index (tdcj_number) is written too, as a plain number
df.to_parquet(out / "session2_clean.parquet")  # and comes back as the index, still a whole number

from_csv = pd.read_csv(out / "session2_clean.csv", dtype=str, keep_default_na=False)
from_parquet = pd.read_parquet(out / "session2_clean.parquet")
print("ID survives the Parquet round trip:", from_parquet.index.name, from_parquet.index.dtype)
print("First TDCJ number as written in the CSV:", from_csv["tdcj_number"].iloc[0], "| padded for people:", cleaning.format_id(df.index[0]))
pd.DataFrame({"missing values": [df.isna().sum().sum(), from_csv.isna().sum().sum(), from_parquet.isna().sum().sum()]},
             index=["in memory", "reloaded from CSV", "reloaded from Parquet"])

A CSV can't tell "missing" apart from "empty text": both are written as nothing between two commas. Reload it carefully and **every null we just created is gone**. Parquet keeps nulls exactly, and after next session it keeps real dates too. Use Parquet (or a Delta table) between steps, and CSV only for handing files to people. When you do, pad the IDs back to 8 digits first with `cleaning.format_id`, or they arrive without their leading zeros. Either way, never save with `index=False` now: the index is the TDCJ number.

## Trust Test check-in

In [ ]:
pd.Series({
    "Q1 Records missing a custody level": df["custody_level"].isna().sum(),
    "Q2 Distinct gender values":          df["gender"].nunique(dropna=False),
    "Q3 Records with last name KENT":     (df["last_name_std"] == "KENT").sum(),
    "Q4 Earliest projected release date": df["projected_release_date"].dropna().min(),
    "Q5 Does TDCJ #02381457 exist?":      cleaning.id_value("02381457") in df.index,
}, name="answer").to_frame()

Four out of five are right now. Q4 is still text sorted alphabetically.

## Wrap-up
- Load untrusted files **as text**, and decide types yourself.
- Codes that mean "unknown" go in a **data dictionary** agreed with the data owner. `NONE` is not always null.
- Repair text damage (encoding, `<br>`, `&nbsp;`) before matching or searching.
- **Match** on standardized values (accents folded); **display** the originals.
- Coded values map through **crosswalks** agreed with the data owner and kept in Git. What a crosswalk doesn't know goes on an exceptions list, never a guess. `N/A` can mean different things in different columns.
- Race and religion are sensitive: standardize what was recorded, restrict access, never infer.
- `tdcj_number` is the **record ID**: prove it (present, 8 digits, unique), convert TDCJ and SID numbers to whole numbers on purpose, then make it the index. Look records up with `id_value`; show IDs to people with `format_id`. Rows aren't people; SIDs are.
- Save with **Parquet**, not CSV.

**Next session:** nine date formats, two-digit years, and dates that parse fine but can't possibly be true.

---
## Answer key
```python
answer_1 = (df["housing_restriction"] == "NONE").sum()
answer_2 = df.loc[df["last_name_std"] == "MCDONALD", "last_name"].nunique()
answer_3 = (records_per_person > 1).sum()
answer_4 = ((df["stg_group"] == "HYDRA") & (df["stg_status"] == "Confirmed")).sum()
```